###Câu a) Đọc bộ dữ liệu Iris

In [1]:
def in_thong_tin():
    print("Họ và tên: Nguyễn Gia Huy")
    print("Mã sinh viên: 6451071027")
    print("-" * 30)

in_thong_tin()

from sklearn.datasets import load_iris

iris = load_iris()
print("data.shape  :", iris.data.shape)
print("target.shape:", iris.target.shape)
print("Các lớp:", iris.target_names)

Họ và tên: Nguyễn Gia Huy
Mã sinh viên: 6451071027
------------------------------
data.shape  : (150, 4)
target.shape: (150,)
Các lớp: ['setosa' 'versicolor' 'virginica']


###Câu b) Chuẩn hóa dữ liệu về [0,1]

In [2]:
in_thong_tin()
import numpy as np

X = iris.data
Y = iris.target

X_max = X.max(axis=0)      # giữ lại để chuẩn hóa mẫu mới ở câu g
X = X / X_max

print("Max từng cột trước chuẩn hóa:", X_max)
print("Min/Max sau chuẩn hóa:", X.min(), X.max())

Họ và tên: Nguyễn Gia Huy
Mã sinh viên: 6451071027
------------------------------
Max từng cột trước chuẩn hóa: [7.9 4.4 6.9 2.5]
Min/Max sau chuẩn hóa: 0.04 1.0


###Câu c) Định nghĩa hàm create_model

In [3]:
in_thong_tin()
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

def create_model():
    model = Sequential([
        Input(shape=(4,)),
        Dense(10, activation='relu'),
        Dense(20, activation='relu'),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

create_model().summary()

Họ và tên: Nguyễn Gia Huy
Mã sinh viên: 6451071027
------------------------------


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 10)             │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            63 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333 (1.30 KB)

 Trainable params: 333 (1.30 KB)

 Non-trainable params: 0 (0.00 B)

###Câu d) Huấn luyện bằng k-fold cross validation (k = 10)

In [4]:
in_thong_tin()
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

models = []
acc_per_fold, pre_per_fold, rec_per_fold, f1_per_fold = [], [], [], []
# Kết quả từng lớp của từng fold (mỗi phần tử là mảng 3 giá trị)
pre_class_per_fold, rec_class_per_fold, f1_class_per_fold = [], [], []

fold_no = 1
for train_index, val_index in kfold.split(X):
    print(f"\n===== Fold {fold_no}/10 =====")

    model = create_model()
    X_train_fold, X_val_fold = X[train_index], X[val_index]
    y_train_fold, y_val_fold = Y[train_index], Y[val_index]

    model.fit(X_train_fold, y_train_fold, epochs=100, batch_size=1, verbose=0)
    models.append(model)

    y_pred_fold = np.argmax(model.predict(X_val_fold, verbose=0), axis=1)

    acc = accuracy_score(y_val_fold, y_pred_fold)
    # Trung bình macro
    pre, rec, f1, _ = precision_recall_fscore_support(
        y_val_fold, y_pred_fold, average='macro', zero_division=0)
    # Từng lớp
    pre_c, rec_c, f1_c, _ = precision_recall_fscore_support(
        y_val_fold, y_pred_fold, labels=[0, 1, 2], zero_division=0)

    acc_per_fold.append(acc);  pre_per_fold.append(pre)
    rec_per_fold.append(rec);  f1_per_fold.append(f1)
    pre_class_per_fold.append(pre_c)
    rec_class_per_fold.append(rec_c)
    f1_class_per_fold.append(f1_c)

    print(f"Accuracy: {acc:.4f} | Precision: {pre:.4f} | Recall: {rec:.4f} | F1-score: {f1:.4f}")
    fold_no += 1

Họ và tên: Nguyễn Gia Huy
Mã sinh viên: 6451071027
------------------------------

===== Fold 1/10 =====
Accuracy: 1.0000 | Precision: 1.0000 | Recall: 1.0000 | F1-score: 1.0000

===== Fold 2/10 =====
Accuracy: 1.0000 | Precision: 1.0000 | Recall: 1.0000 | F1-score: 1.0000

===== Fold 3/10 =====
Accuracy: 1.0000 | Precision: 1.0000 | Recall: 1.0000 | F1-score: 1.0000

===== Fold 4/10 =====
Accuracy: 0.9333 | Precision: 0.9524 | Recall: 0.9333 | F1-score: 0.9373

===== Fold 5/10 =====


Accuracy: 0.9333 | Precision: 0.9333 | Recall: 0.9333 | F1-score: 0.9259

===== Fold 6/10 =====


Accuracy: 0.8667 | Precision: 0.8667 | Recall: 0.8889 | F1-score: 0.8500

===== Fold 7/10 =====
Accuracy: 0.8667 | Precision: 0.9167 | Recall: 0.8333 | F1-score: 0.8413

===== Fold 8/10 =====
Accuracy: 1.0000 | Precision: 1.0000 | Recall: 1.0000 | F1-score: 1.0000

===== Fold 9/10 =====
Accuracy: 1.0000 | Precision: 1.0000 | Recall: 1.0000 | F1-score: 1.0000

===== Fold 10/10 =====
Accuracy: 1.0000 | Precision: 1.0000 | Recall: 1.0000 | F1-score: 1.0000


###Câu e) Average Accuracy, Recall, Precision, F1-score

In [5]:
in_thong_tin()

print(f"Average Accuracy : {np.mean(acc_per_fold):.4f}")
print(f"Average Precision: {np.mean(pre_per_fold):.4f}")
print(f"Average Recall   : {np.mean(rec_per_fold):.4f}")
print(f"Average F1-score : {np.mean(f1_per_fold):.4f}")

Họ và tên: Nguyễn Gia Huy
Mã sinh viên: 6451071027
------------------------------
Average Accuracy : 0.9600
Average Precision: 0.9669
Average Recall   : 0.9589
Average F1-score : 0.9555


###Câu f) Average Recall, Precision, F1-score cho từng class

In [6]:
in_thong_tin()
import pandas as pd

ket_qua = pd.DataFrame({
    'Class': iris.target_names,
    'Avg Precision': np.mean(pre_class_per_fold, axis=0),
    'Avg Recall':    np.mean(rec_class_per_fold, axis=0),
    'Avg F1-score':  np.mean(f1_class_per_fold, axis=0),
})
print(ket_qua.round(4).to_string(index=False))

Họ và tên: Nguyễn Gia Huy
Mã sinh viên: 6451071027
------------------------------
     Class  Avg Precision  Avg Recall  Avg F1-score
    setosa         1.0000      1.0000        1.0000
versicolor         0.9657      0.9167        0.9279
 virginica         0.9350      0.9600        0.9385


###Câu g) Dự đoán 3 mẫu mới bằng mô hình có F1-score tốt nhất

In [7]:
in_thong_tin()

best_index = np.argmax(f1_per_fold)
best_model = models[best_index]
print(f"Mô hình tốt nhất: fold {best_index + 1} (F1-score = {f1_per_fold[best_index]:.4f})")

X_new = np.array([[6.2, 2.9, 4.3, 1.3],
                  [5.1, 3.5, 1.4, 0.2],
                  [7.3, 2.8, 6.4, 2.1]])
X_new_scaled = X_new / X_max

y_prob_new = best_model.predict(X_new_scaled)
y_pred_new = np.argmax(y_prob_new, axis=1)

for mau, p, prob in zip(X_new, y_pred_new, y_prob_new):
    print(f"Mẫu {mau.tolist()} -> {iris.target_names[p]} (xác suất {prob.max():.4f})")

Họ và tên: Nguyễn Gia Huy
Mã sinh viên: 6451071027
------------------------------
Mô hình tốt nhất: fold 1 (F1-score = 1.0000)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step
Mẫu [6.2, 2.9, 4.3, 1.3] -> versicolor (xác suất 0.9994)
Mẫu [5.1, 3.5, 1.4, 0.2] -> setosa (xác suất 1.0000)
Mẫu [7.3, 2.8, 6.4, 2.1] -> virginica (xác suất 0.9997)
